In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
print(sys.executable)
import pandas as pd
print("Pandas:", pd.__version__)
print("✅ Ready for Phase 1!")

In [ ]:
# ================================
# PHASE 1: DATA INGESTION
# IRS-1: AI-Driven Demand Intelligence
# ================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── 1A. Load FreshRetailNet-50K ──────────────────────────────────
# If downloaded as parquet from HuggingFace:
fresh_df = pd.read_parquet('data/raw/freshretailnet/train.parquet')

# If downloaded as CSV:
# fresh_df = pd.read_csv('data/raw/freshretailnet/fresh_retail.csv')

print("FreshRetailNet Shape:", fresh_df.shape)
print("FreshRetailNet Columns:\n", fresh_df.columns.tolist())
print("\nSample:\n", fresh_df.head(3))
print("\nData Types:\n", fresh_df.dtypes)
print("\nNull Values:\n", fresh_df.isnull().sum())

In [ ]:
import sys
print(sys.executable)


In [ ]:
import os
print("Current Working Directory:", os.getcwd())
print("\nFiles in current folder:")
for item in os.listdir('.'):
    print(item)
    

In [ ]:


# It might be this:
fresh_df = pd.read_parquet('C:/Users/Dell/Desktop/Naveed/gitdemo/forecasting-future/data/raw/freshretailnet/train.parquet')

In [ ]:
import os

print("Current Directory:", os.getcwd())
print("\nSearching for train.parquet...\n")
for root, dirs, files in os.walk('C:/Users/Dell/Desktop/Naveed/gitdemo/forecasting-future'):
    for file in files:
        if file.endswith('.parquet'):
            print(os.path.join(root, file))

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Fix base path — go one level up from notebooks folder
BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/forecasting-future'

# ── 1A. Load FreshRetailNet-50K ──────────────────────────────────
fresh_df = pd.read_parquet(f'{BASE_PATH}/data/raw/freshretailnet/train.parquet')

print("✅ FreshRetailNet Loaded!")
print("Shape:", fresh_df.shape)
print("Columns:", fresh_df.columns.tolist())
print("\nSample:")
print(fresh_df.head(3))

In [ ]:
import os

print("Searching for Kaggle dataset...\n")
for root, dirs, files in os.walk('C:/Users/Dell/Desktop/Naveed/gitdemo/forecasting-future'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

In [ ]:
# ── 1C. Load Kaggle Inventory Optimization Dataset ───────────────
BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/forecasting-future'

# Load all 3 files
demand_df = pd.read_csv(f'{BASE_PATH}/data/raw/kaggle/demand_forecasting.csv')
inventory_df = pd.read_csv(f'{BASE_PATH}/data/raw/kaggle/inventory_monitoring.csv')
pricing_df = pd.read_csv(f'{BASE_PATH}/data/raw/kaggle/pricing_optimization.csv')

# ── Inspect demand_forecasting.csv ───────────────────────────────
print("=" * 50)
print("📦 DEMAND FORECASTING FILE")
print("=" * 50)
print("Shape:", demand_df.shape)
print("Columns:", demand_df.columns.tolist())
print(demand_df.head(3))

# ── Inspect inventory_monitoring.csv ─────────────────────────────
print("\n" + "=" * 50)
print("📦 INVENTORY MONITORING FILE")
print("=" * 50)
print("Shape:", inventory_df.shape)
print("Columns:", inventory_df.columns.tolist())
print(inventory_df.head(3))

# ── Inspect pricing_optimization.csv ─────────────────────────────
print("\n" + "=" * 50)
print("📦 PRICING OPTIMIZATION FILE")
print("=" * 50)
print("Shape:", pricing_df.shape)
print("Columns:", pricing_df.columns.tolist())
print(pricing_df.head(3))

In [ ]:
# ── DATASET HEALTH CHECK ─────────────────────────────────────────

print("=" * 50)
print("🔍 FRESHRETAILNET HEALTH CHECK")
print("=" * 50)
print("Shape:", fresh_df.shape)
print("\nNull Values:")
print(fresh_df.isnull().sum())

# Fix: use only basic scalar columns for duplicate check
scalar_cols = ['city_id', 'store_id', 'product_id', 'dt', 
               'sale_amount', 'discount', 'holiday_flag', 
               'activity_flag', 'precpt', 'avg_temperature',
               'avg_humidity', 'avg_wind_level']
print("\nDuplicate Rows:", fresh_df[scalar_cols].duplicated().sum())
print("\nDate Range:", fresh_df['dt'].min(), "→", fresh_df['dt'].max())

print("\n" + "=" * 50)
print("🔍 KAGGLE DEMAND FILE HEALTH CHECK")
print("=" * 50)
print("Shape:", demand_df.shape)
print("\nNull Values:")
print(demand_df.isnull().sum())
print("\nDuplicate Rows:", demand_df.duplicated().sum())
print("\nDate Range:", demand_df['Date'].min(), 
      "→", demand_df['Date'].max())

print("\n" + "=" * 50)
print("🔍 KAGGLE INVENTORY FILE HEALTH CHECK")
print("=" * 50)
print("Shape:", inventory_df.shape)
print("\nNull Values:")
print(inventory_df.isnull().sum())
print("\nDuplicate Rows:", inventory_df.duplicated().sum())

print("\n" + "=" * 50)
print("🔍 KAGGLE PRICING FILE HEALTH CHECK")
print("=" * 50)
print("Shape:", pricing_df.shape)
print("\nNull Values:")
print(pricing_df.isnull().sum())
print("\nDuplicate Rows:", pricing_df.duplicated().sum())

In [ ]:
# ── FRESHRETAILNET PREPROCESSING ────────────────────────────────
print("Starting FreshRetailNet Preprocessing...")

fresh = fresh_df.copy()

# ── PARSING: Convert dt to datetime ─────────────────────────────
fresh['dt'] = pd.to_datetime(fresh['dt'])

# ── PARSING: Extract time features ──────────────────────────────
fresh['year']        = fresh['dt'].dt.year
fresh['month']       = fresh['dt'].dt.month
fresh['day']         = fresh['dt'].dt.day
fresh['week']        = fresh['dt'].dt.isocalendar().week.astype(int)
fresh['day_of_week'] = fresh['dt'].dt.dayofweek
fresh['is_weekend']  = (fresh['day_of_week'] >= 5).astype(int)
fresh['quarter']     = fresh['dt'].dt.quarter

# ── Drop array columns (not needed for modeling) ─────────────────
fresh = fresh.drop(columns=['hours_sale', 'hours_stock_status'])

# ── NORMALIZATION: Scale numeric columns ─────────────────────────
from sklearn.preprocessing import MinMaxScaler

scaler_fresh = MinMaxScaler()
num_cols_fresh = ['sale_amount', 'stock_hour6_22_cnt', 
                  'precpt', 'avg_temperature', 
                  'avg_humidity', 'avg_wind_level']

fresh[num_cols_fresh] = scaler_fresh.fit_transform(fresh[num_cols_fresh])

print("✅ FreshRetailNet Preprocessing Done!")
print("Shape after preprocessing:", fresh.shape)
print("New columns added:", ['year','month','day',
                             'week','day_of_week',
                             'is_weekend','quarter'])
print("\nSample:")
print(fresh[['store_id', 'product_id', 'dt', 'sale_amount',
             'year', 'month', 'is_weekend']].head(3))

In [ ]:
import sys
print(sys.executable)

In [ ]:
# ── KAGGLE PREPROCESSING ─────────────────────────────────────────
print("\nStarting Kaggle Datasets Preprocessing...")

# ── Copy all 3 files ─────────────────────────────────────────────
demand    = demand_df.copy()
inventory = inventory_df.copy()
pricing   = pricing_df.copy()

# ── PARSING: Convert Date column ─────────────────────────────────
demand['Date'] = pd.to_datetime(demand['Date'])
demand['year']        = demand['Date'].dt.year
demand['month']       = demand['Date'].dt.month
demand['day']         = demand['Date'].dt.day
demand['day_of_week'] = demand['Date'].dt.dayofweek
demand['is_weekend']  = (demand['day_of_week'] >= 5).astype(int)
demand['quarter']     = demand['Date'].dt.quarter

# ── ENCODING: Convert categorical columns to numeric ──────────────
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
cat_cols = ['Promotions', 'Seasonality Factors', 
            'External Factors', 'Demand Trend', 
            'Customer Segments']

for col in cat_cols:
    demand[col + '_encoded'] = le.fit_transform(demand[col])

# ── NORMALIZATION: Scale numeric columns ──────────────────────────
scaler_demand = MinMaxScaler()
num_cols_demand = ['Sales Quantity', 'Price']
demand[num_cols_demand] = scaler_demand.fit_transform(
                          demand[num_cols_demand])

scaler_inv = MinMaxScaler()
num_cols_inv = ['Stock Levels', 'Supplier Lead Time (days)',
                'Stockout Frequency', 'Reorder Point',
                'Warehouse Capacity', 
                'Order Fulfillment Time (days)']
inventory[num_cols_inv] = scaler_inv.fit_transform(
                          inventory[num_cols_inv])

scaler_price = MinMaxScaler()
num_cols_price = ['Storage Cost', 'Elasticity Index',
                  'Return Rate (%)']

# Only scale columns that exist
num_cols_price = [c for c in num_cols_price if c in pricing.columns]
pricing[num_cols_price] = scaler_price.fit_transform(
                          pricing[num_cols_price])

print("✅ Kaggle Preprocessing Done!")
print("Demand shape:",    demand.shape)
print("Inventory shape:", inventory.shape)
print("Pricing shape:",   pricing.shape)

In [ ]:
# ── MERGE KAGGLE FILES ───────────────────────────────────────────
print("\nMerging Kaggle files...")

# Merge demand + inventory on Product ID and Store ID
kaggle_merged = demand.merge(
    inventory,
    on=['Product ID', 'Store ID'],
    how='left'
)

# Merge with pricing on Product ID
kaggle_merged = kaggle_merged.merge(
    pricing,
    on='Product ID',
    how='left'
)

print("✅ Kaggle Merge Done!")
print("Unified Kaggle Shape:", kaggle_merged.shape)
print("Columns:", kaggle_merged.columns.tolist())
print("\nSample:")
print(kaggle_merged.head(3))

In [ ]:
# ── SAVE PROCESSED FILES ─────────────────────────────────────────
import os
os.makedirs(f'{BASE_PATH}/data/processed', exist_ok=True)

fresh.to_csv(
    f'{BASE_PATH}/data/processed/fresh_processed.csv', 
    index=False)

kaggle_merged.to_csv(
    f'{BASE_PATH}/data/processed/kaggle_unified.csv',  
    index=False)

print("✅ All files saved!")
print(f"   fresh_processed.csv  → {fresh.shape}")
print(f"   kaggle_unified.csv   → {kaggle_merged.shape}")
print("\n🎉 PHASE 1 COMPLETE!")

In [ ]:
# ════════════════════════════════════════
# PHASE 1 FIX — Correct encoding + 
# Remove scaled values
# ════════════════════════════════════════
import pandas as pd
import numpy as np

BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'

# ── Reload original RAW demand file ──────────────────────────────
# We go back to raw because kaggle_unified has scaled values
# which is the data leakage bug

demand_raw = pd.read_csv(
    f'{BASE_PATH}/data/raw/kaggle/demand_forecasting.csv'
)
inventory_raw = pd.read_csv(
    f'{BASE_PATH}/data/raw/kaggle/inventory_monitoring.csv'
)
pricing_raw = pd.read_csv(
    f'{BASE_PATH}/data/raw/kaggle/pricing_optimization.csv'
)

print("Raw files loaded:")
print(f"  Demand    : {demand_raw.shape}")
print(f"  Inventory : {inventory_raw.shape}")
print(f"  Pricing   : {pricing_raw.shape}")

# ── Merge exactly like Phase 1 but WITHOUT scaling ───────────────
kaggle_clean = demand_raw.merge(
    inventory_raw,
    on  = ['Product ID', 'Store ID'],
    how = 'left'
)
kaggle_clean = kaggle_clean.merge(
    pricing_raw,
    on  = 'Product ID',
    how = 'left'
)

print(f"\nMerged shape: {kaggle_clean.shape}")

# ── Add time features ─────────────────────────────────────────────
kaggle_clean['Date']        = pd.to_datetime(kaggle_clean['Date'])
kaggle_clean['year']        = kaggle_clean['Date'].dt.year
kaggle_clean['month']       = kaggle_clean['Date'].dt.month
kaggle_clean['day']         = kaggle_clean['Date'].dt.day
kaggle_clean['day_of_week'] = kaggle_clean['Date'].dt.dayofweek
kaggle_clean['is_weekend']  = (kaggle_clean['day_of_week'] >= 5).astype(int)
kaggle_clean['quarter']     = kaggle_clean['Date'].dt.quarter

# ── Drop old LabelEncoded columns ────────────────────────────────
drop_encoded = [c for c in kaggle_clean.columns 
                if '_encoded' in c]
if drop_encoded:
    kaggle_clean = kaggle_clean.drop(columns=drop_encoded)
    print(f"\nDropped old encoded columns: {drop_encoded}")

# ── OneHot encode categorical columns ────────────────────────────
# These have NO natural order — OneHot is correct
ohe_cols = ['Promotions', 'Seasonality Factors',
            'External Factors', 'Demand Trend',
            'Customer Segments']
ohe_cols = [c for c in ohe_cols if c in kaggle_clean.columns]

print(f"\nOneHot encoding: {ohe_cols}")
for col in ohe_cols:
    print(f"  {col}: {kaggle_clean[col].unique()}")

kaggle_encoded = pd.get_dummies(
    kaggle_clean,
    columns   = ohe_cols,
    drop_first = False
)

print(f"\nShape before encoding: {kaggle_clean.shape}")
print(f"Shape after encoding:  {kaggle_encoded.shape}")

# ── Verify sales are NOT scaled ───────────────────────────────────
print(f"\nSales Quantity sample values (should be real numbers):")
print(kaggle_encoded['Sales Quantity'].head(5).tolist())
print(f"Min: {kaggle_encoded['Sales Quantity'].min()}")
print(f"Max: {kaggle_encoded['Sales Quantity'].max()}")

# ── Save corrected version ────────────────────────────────────────
kaggle_encoded.to_csv(
    f'{BASE_PATH}/data/processed/kaggle_unified_encoded.csv',
    index=False
)

print(f"\n✅ Saved kaggle_unified_encoded.csv → {kaggle_encoded.shape}")
print("\n⚠️  NOTE: NO scaling applied")
print("Sales and Price are now in ORIGINAL units")
print("Scaling will happen in Phase 3 AFTER train-test split")

In [ ]:
# Quick check — see all column names in kaggle_unified.csv
import pandas as pd
BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'

kaggle = pd.read_csv(f'{BASE_PATH}/data/processed/kaggle_unified.csv')
print("All columns in kaggle_unified.csv:")
for i, col in enumerate(kaggle.columns):
    print(f"  {i+1}. {col}  —  dtype: {kaggle[col].dtype}  —  sample: {kaggle[col].iloc[0]}")

In [ ]:
import pandas as pd
BASE_PATH = 'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'

check = pd.read_csv(f'{BASE_PATH}/data/processed/kaggle_unified_encoded.csv')
print("Shape:", check.shape)
print("\nSales Quantity sample:")
print(check['Sales Quantity'].head(5).tolist())
print(f"Min: {check['Sales Quantity'].min()}")
print(f"Max: {check['Sales Quantity'].max()}")
print(f"\nNew OneHot columns created:")
ohe_new = [c for c in check.columns if any(
    x in c for x in ['Promotions_', 'Seasonality_', 
                      'External_', 'Demand_Trend_', 
                      'Customer_'])]
for c in ohe_new:
    print(f"  {c}")

In [ ]:
# ════════════════════════════════════════
# PHASE 1 FIX CELL 2 — Create config.py
# Fixes all hardcoded paths
# ════════════════════════════════════════

config_content = '''\
# ================================
# config.py — Project Configuration
# IRS-1: AI-Driven Demand Intelligence
# ================================
from pathlib import Path

# ── Auto-detect project root ──────────────────────────────────────
# Works on ANY computer — no hardcoded paths needed
PROJECT_ROOT = Path(__file__).resolve().parent

# ── Data paths ────────────────────────────────────────────────────
RAW_PATH       = PROJECT_ROOT / "data" / "raw"
KAGGLE_PATH    = RAW_PATH / "kaggle"
FRESH_PATH     = RAW_PATH / "freshretailnet"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
MODELS_PATH    = PROJECT_ROOT / "models"

# ── Key processed files ───────────────────────────────────────────
KAGGLE_ENCODED   = PROCESSED_PATH / "kaggle_unified_encoded.csv"
FRESH_PROCESSED  = PROCESSED_PATH / "fresh_processed.csv"
DEMAND_FEATURES  = PROCESSED_PATH / "demand_features_v2.csv"
PRODUCT_SEGMENTS = PROCESSED_PATH / "product_segments.csv"
FEATURE_COLUMNS  = PROCESSED_PATH / "feature_columns.json"
MODEL_RESULTS    = PROCESSED_PATH / "model_results.csv"

# ── Create folders if they do not exist ───────────────────────────
for path in [RAW_PATH, KAGGLE_PATH, FRESH_PATH,
             PROCESSED_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)
'''

# ── Save config.py to project root ───────────────────────────────
import os
BASE_PATH   = 'C:/Users/Dell/Desktop/Naveed/gitdemo/Forecasting-future'
config_path = f'{BASE_PATH}/config.py'

with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_content)

print(f"✅ config.py saved to project root")
print(f"   Path: {config_path}")

# ── Test it immediately ───────────────────────────────────────────
import sys
sys.path.append(BASE_PATH)
from config import (PROJECT_ROOT, PROCESSED_PATH, 
                    MODELS_PATH, DEMAND_FEATURES,
                    KAGGLE_ENCODED)

print(f"\n✅ config.py import test passed")
print(f"   PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"   PROCESSED_PATH : {PROCESSED_PATH}")
print(f"   MODELS_PATH    : {MODELS_PATH}")
print(f"\nFrom now on use this in every notebook:")
print("   import sys")
print(f"   sys.path.append('{BASE_PATH}')")
print("   from config import PROCESSED_PATH, MODELS_PATH")
print("\n✅ Phase 1 fixes complete!")